# Climate Data Analysis

You're working with historical climate data: global temperatures, CO2 concentrations, and sea levels going back to 1850. In this lab, you'll build a regression model on this data, evaluate it properly, and see first-hand what happens when you push a model beyond what it's seen.

By the end, you'll have a concrete understanding of error metrics, the difference between interpolation and extrapolation, and why a model that fits the training data perfectly can be worse than useless.

## Loading and inspecting the data

Let's start by loading the dataset and getting a feel for what we're working with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("./climate_data.csv")
df.head(10)

The dataset has four columns:

- **year**: the observation year (1850–2020)
- **co2_ppm**: atmospheric CO2 concentration in parts per million
- **global_mean_temp_anomaly**: how much warmer or cooler that year was compared to a baseline average (in °C)
- **sea_level_mm**: cumulative sea level change in millimetres

Temperature anomaly is measured relative to a baseline period (typically a 20th-century average). A value of +0.5 means that year was half a degree warmer than the baseline. A negative value means it was cooler.

In [ ]:
df.describe()

## Exploring the data over time

Before building any model, we should look at the data. Two of the most important columns are temperature anomaly and CO2 concentration. Let's plot them over time and see what patterns emerge.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(df["year"], df["global_mean_temp_anomaly"], linewidth=1)
ax1.set_ylabel("Temperature anomaly (°C)")
ax1.set_title("Global mean temperature anomaly, 1850–2020")

ax2.plot(df["year"], df["co2_ppm"], linewidth=1, color="tab:orange")
ax2.set_ylabel("CO2 (ppm)")
ax2.set_xlabel("Year")
ax2.set_title("Atmospheric CO2 concentration, 1850–2020")

plt.tight_layout()
plt.show()

Both variables trend upward, particularly after 1950. CO2 rises steeply from around 310 ppm to over 410 ppm. Temperature anomaly is noisier year-to-year but shows a clear warming trend.

The fact that both are rising doesn't prove one causes the other (that requires physics, not just statistics), but it does suggest a strong relationship. Let's see whether that holds up in a scatter plot.

## Measuring the relationship

If CO2 and temperature anomaly are related, we should see it when we plot one against the other.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["co2_ppm"], df["global_mean_temp_anomaly"], alpha=0.5, s=15)
plt.xlabel("CO2 (ppm)")
plt.ylabel("Temperature anomaly (°C)")
plt.title("CO2 concentration vs temperature anomaly")
plt.tight_layout()
plt.show()

The points form a rough upward trend: higher CO2 tends to coincide with higher temperature anomalies. But there's scatter, especially at lower CO2 values where the temperature bounces around quite a bit.

We can put a number on how strong this linear relationship is using the **correlation coefficient** (often written as *r*). It ranges from -1 to +1:

- **+1** means a perfect positive linear relationship (as one goes up, the other goes up in lockstep)
- **0** means no linear relationship at all
- **-1** means a perfect negative linear relationship

Values above 0.7 or below -0.7 are generally considered strong. Let's calculate it.

In [ ]:
correlation = df["co2_ppm"].corr(df["global_mean_temp_anomaly"])
print(f"Correlation between CO2 and temperature anomaly: {correlation:.3f}")

That's a strong positive correlation. A linear model has something real to work with here. Let's build one.

## Building a linear regression

We want to predict temperature anomaly from CO2 concentration. Before fitting the model, we split the data into a **training set** (80%) and a **test set** (20%). The model learns from the training set. We evaluate it on the test set, which it has never seen, to check whether it generalises or has just memorised the training data.

In [ ]:
X = df[["co2_ppm"]]
y = df["global_mean_temp_anomaly"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} rows")
print(f"Test set:     {len(X_test)} rows")

Now we fit the model. When you call `.fit()`, scikit-learn finds the straight line that minimises the total squared error across all training points. The result is two numbers: a **slope** (how much the temperature changes per unit of CO2) and an **intercept** (where the line crosses the y-axis).

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Slope:     {model.coef_[0]:.5f} °C per ppm")
print(f"Intercept: {model.intercept_:.4f} °C")

## How good is the model?

Let's see how the model's predictions compare to reality on the test set. The plot below shows the regression line, the actual data points, and the **residuals**: vertical lines connecting each point to the line. Each residual is the error for one prediction.

In [ ]:
y_pred = model.predict(X_test)

plt.figure(figsize=(10, 5))

# Regression line across the full CO2 range
co2_range = np.linspace(X_test["co2_ppm"].min(), X_test["co2_ppm"].max(), 200)
plt.plot(co2_range, model.predict(co2_range.reshape(-1, 1)),
         color="red", linewidth=2, label="Regression line")

# Actual data points
plt.scatter(X_test["co2_ppm"], y_test, alpha=0.7, zorder=3, label="Actual")

# Residuals
for co2, actual, predicted in zip(X_test["co2_ppm"], y_test, y_pred):
    plt.plot([co2, co2], [predicted, actual], color="grey", linewidth=0.8, alpha=0.6)

plt.xlabel("CO2 (ppm)")
plt.ylabel("Temperature anomaly (°C)")
plt.title("Residuals: the gap between the line and the data")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"MAE:  {mae:.4f} °C")
print(f"RMSE: {rmse:.4f} °C")
print(f"MSE:  {mse:.6f} °C²")

Three ways to measure the same thing: how wrong the model is.

- **MAE** (Mean Absolute Error) is the average size of the mistakes, ignoring direction. If the MAE is 0.1°C, the model is off by about a tenth of a degree on average.
- **RMSE** (Root Mean Squared Error) is similar but penalises large errors more heavily. If MAE and RMSE are close together, the errors are fairly uniform. If RMSE is noticeably larger, there are some big outlier predictions dragging it up.
- **MSE** (Mean Squared Error) is the average of the squared errors. The units aren't intuitive (°C²), but MSE is what the model actually minimises during training. RMSE is just the square root of MSE, which brings it back to the original units.

Is this model good enough? That depends entirely on what the predictions are for. For a rough trend analysis, these numbers might be fine. For precise climate projections informing policy decisions, you'd want much more scrutiny.

## The loss function

During training, the model tries different lines and measures how wrong each one is using **MSE** as its **loss function**. The line that produces the lowest MSE wins.

Why squared errors rather than absolute errors? Squaring makes big mistakes count for much more than small ones. Look at how fast the penalty grows:

In [ ]:
print("Error (°C)  |  Squared (contribution to loss)")
print("------------|----------------------------------")
for error in [0.25, 0.5, 1.0, 2.0, 4.0]:
    print(f"  {error:.2f}       |  {error**2:.2f}")

A prediction that's off by 2°C contributes 16 times as much to the loss as one that's off by 0.5°C. The model is strongly incentivised to avoid large errors, even if it means tolerating a few small ones.

## Interpolation vs extrapolation

Our model was trained on CO2 values roughly between 280 and 413 ppm. What happens when we ask it to predict within that range versus far beyond it?

In [ ]:
co2_min = df["co2_ppm"].min()
co2_max = df["co2_ppm"].max()

# Extend well beyond the training range
co2_full = np.linspace(250, 800, 500).reshape(-1, 1)
temp_pred = model.predict(co2_full)

plt.figure(figsize=(12, 5))

# Shade the training range
plt.axvspan(co2_min, co2_max, alpha=0.1, color="green", label="Training range")

# Observed data
plt.scatter(df["co2_ppm"], df["global_mean_temp_anomaly"],
            alpha=0.4, s=15, zorder=3, label="Observed data")

# Regression line (full range, including extrapolation)
plt.plot(co2_full, temp_pred, color="red", linewidth=2, label="Model prediction")

# Mark specific predictions
for ppm, marker, colour in [(354, "o", "green"), (550, "s", "red"), (800, "^", "purple")]:
    pred = model.predict([[ppm]])[0]
    label = f"{ppm} ppm: {pred:+.2f}°C"
    plt.plot(ppm, pred, marker=marker, color=colour, markersize=10, zorder=4, label=label)

plt.xlabel("CO2 (ppm)")
plt.ylabel("Temperature anomaly (°C)")
plt.title("Interpolation vs extrapolation")
plt.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

The green-shaded region is where the model has evidence. The prediction at 354 ppm (roughly 1990) is **interpolation**: the model has seen similar values and can make a reasonable estimate.

The predictions at 550 and 800 ppm are **extrapolation**: the model is extending its straight line into territory it has never seen. It has no way to know whether the relationship stays linear at those concentrations. Climate systems have tipping points, feedback loops, and non-linearities that a straight line cannot capture.

The model will happily produce a number for any input you give it. It won't warn you when that number is a guess. That's your job.

## Overfitting: when the model learns too much

A straight line might be too simple for this data. What if we used a more flexible curve? Polynomial regression lets us fit curves of increasing complexity: degree 1 is a straight line, degree 3 is a cubic, degree 15 can wiggle through almost any set of points.

We'll extend the x-axis well beyond the training data so you can see how each model behaves in the unknown.

In [ ]:
X_simple = df[["year"]].values
y_simple = df["global_mean_temp_anomaly"].values

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

degrees = [1, 3, 5, 15]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extend well beyond the training range
X_plot = np.linspace(1820, 2080, 500).reshape(-1, 1)

for ax, degree in zip(axes.flatten(), degrees):
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_s)
    X_test_poly = poly.transform(X_test_s)
    X_plot_poly = poly.transform(X_plot)

    lr = LinearRegression()
    lr.fit(X_train_poly, y_train_s)

    train_rmse = np.sqrt(mean_squared_error(y_train_s, lr.predict(X_train_poly)))
    test_rmse = np.sqrt(mean_squared_error(y_test_s, lr.predict(X_test_poly)))

    # Shade the training data range
    ax.axvspan(df["year"].min(), df["year"].max(), alpha=0.08, color="green")

    ax.scatter(X_train_s, y_train_s, alpha=0.3, s=8, label="Train")
    ax.scatter(X_test_s, y_test_s, alpha=0.3, s=8, label="Test")
    ax.plot(X_plot, lr.predict(X_plot_poly), color="red", linewidth=2)
    ax.set_title(f"Degree {degree}\nTrain RMSE: {train_rmse:.3f} | Test RMSE: {test_rmse:.3f}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Temp anomaly (°C)")
    ax.set_ylim(-3, 5)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Degree':<10} {'Train RMSE':<15} {'Test RMSE':<15} {'Gap':<10}")
print("-" * 50)
for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_s)
    X_test_poly = poly.transform(X_test_s)
    lr = LinearRegression()
    lr.fit(X_train_poly, y_train_s)
    train_rmse = np.sqrt(mean_squared_error(y_train_s, lr.predict(X_train_poly)))
    test_rmse = np.sqrt(mean_squared_error(y_test_s, lr.predict(X_test_poly)))
    print(f"{degree:<10} {train_rmse:<15.4f} {test_rmse:<15.4f} {test_rmse - train_rmse:<10.4f}")

Look at the degree-15 plot. Inside the green-shaded training range, it hugs the data beautifully. Outside it, the curve rockets off to absurd values. This is **overfitting**: the model has memorised the noise in the training data rather than learning the underlying trend.

The RMSE table tells the same story numerically. As the degree increases, training RMSE keeps falling (the model fits the training data better and better). But test RMSE first improves then gets worse. The growing gap between train and test RMSE is the signature of overfitting.

Degree 3 or 5 strikes a reasonable balance: flexible enough to capture the curve in the data, but not so flexible that it hallucinates patterns that aren't there. Finding this balance is one of the central challenges in machine learning.